In [1]:

# Matplotlib
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
# Numpy
import numpy as np
# Pandas
import pandas as pd
# Torch
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchmetrics.classification import BinaryAccuracy
# Helper functions (additional file)
from helper_functions import *

In [2]:
# Use GPU if available, else use CPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

cuda


In [3]:
# Columns for dataset
filepath = 'creditcard.csv'
df = pd.read_csv(filepath)
print(df.columns)

# Load the dataset
V1, V2, V3, V4, V5, V6, V7, V8, V9, V10, V11, V12, V13, V14, V15, V16, V17, V18, V19, V20, V21, V22, V23, V24, V25, V26, V27, V28, amount, inputs, outputs = load_creditcard_dataset(filepath)

print("Input shape:", inputs.shape)
print("Output shape:", outputs.shape)
print("Number of samples with class 0:", len(outputs) - sum(outputs))
print("Number of samples with class 1:", sum(outputs))

print()


Index(['Time', 'V1', 'V2', 'V3', 'V4', 'V5', 'V6', 'V7', 'V8', 'V9', 'V10',
       'V11', 'V12', 'V13', 'V14', 'V15', 'V16', 'V17', 'V18', 'V19', 'V20',
       'V21', 'V22', 'V23', 'V24', 'V25', 'V26', 'V27', 'V28', 'Amount',
       'Class'],
      dtype='object')
Input shape: (284807, 28)
Output shape: (284807,)
Number of samples with class 0: 284315
Number of samples with class 1: 492



In [4]:

class CreditCardDataset(Dataset):
    def __init__(self):
        self.dataframe = pd.read_csv(filepath)
        
    def __len__(self):
        return len(self.dataframe) 
    
    def __getitem__(self, idx):
        # Extract features and target from the dataframe at index idx
        V1 = self.dataframe.iloc[idx]['V1']      # SOLUTION: Get x1 value
        V2 = self.dataframe.iloc[idx]['V2']      # SOLUTION: Get x2 value
        y = self.dataframe.iloc[idx]['Class']    # SOLUTION: Get class value
        
        # Convert to PyTorch tensors
        V1 = torch.tensor(V1, dtype=torch.float32)
        V2 = torch.tensor(V2, dtype=torch.float32)
        V3 = torch.tensor(V3, dtype=torch.float32)
        V4 = torch.tensor(V4, dtype=torch.float32)
        V5 = torch.tensor(V5, dtype=torch.float32)
        V6 = torch.tensor(V6, dtype=torch.float32)
        V7 = torch.tensor(V7, dtype=torch.float32)
        V8 = torch.tensor(V8, dtype=torch.float32)
        V9 = torch.tensor(V9, dtype=torch.float32)
        V10 = torch.tensor(V10, dtype=torch.float32)
        V11 = torch.tensor(V11, dtype=torch.float32)
        V12 = torch.tensor(V12, dtype=torch.float32)
        V13 = torch.tensor(V13, dtype=torch.float32)
        V14 = torch.tensor(V14, dtype=torch.float32)
        V15 = torch.tensor(V15, dtype=torch.float32)
        V16 = torch.tensor(V16, dtype=torch.float32)
        V17 = torch.tensor(V17, dtype=torch.float32)
        V18 = torch.tensor(V18, dtype=torch.float32)
        V19 = torch.tensor(V19, dtype=torch.float32)
        V20 = torch.tensor(V20, dtype=torch.float32)
        V21 = torch.tensor(V21, dtype=torch.float32)
        V22 = torch.tensor(V22, dtype=torch.float32)
        V23 = torch.tensor(V23, dtype=torch.float32)
        V24 = torch.tensor(V24, dtype=torch.float32)
        V25 = torch.tensor(V25, dtype=torch.float32)
        V26 = torch.tensor(V26, dtype=torch.float32)
        V27 = torch.tensor(V27, dtype=torch.float32)
        V28 = torch.tensor(V28, dtype=torch.float32)
        amount = torch.tensor(amount, dtype=torch.float32)
        y = torch.tensor(y, dtype=torch.float32)
        
        # Stack features into a single input tensor
        # Combine x1 and x2 into a tensor of shape (2,)
        inputs = torch.stack([V1, V2, V3, V4 , V5, V6, V7, V8, V9, V10, V11, V12, V13, V14, V15, V16, V17, V18, V19, V20, V21, V22, V23, V24, V25, V26, V27, V28, amount])  # SOLUTION: Stack all features into a tensor of shape (29,)
        
        return inputs, y

In [ ]:
cc_dataset  = CreditCardDataset()
batch_size = 64
cc_dataloader = DataLoader(cc_dataset, batch_size=batch_size, shuffle=True)

In [7]:
#Creation of a gated layer
class GatedLayer(nn.Module):
    def __init__(self, input_size, output_size):
        super(GatedLayer, self).__init__()
        self.linear = nn.Linear(input_size, output_size)  # Linear transformation
        self.gate = nn.Linear(input_size, output_size)    # Gate transformation
        self.sigmoid = nn.Sigmoid()                       # Sigmoid activation for gate

    def forward(self, x):
        linear_output = self.linear(x)                    # Compute linear output
        gate_output = self.sigmoid(self.gate(x))         # Compute gate output and apply sigmoid
        return linear_output * gate_output                # Element-wise multiplication of linear output and gate output

In [ ]:
#Architecture of the model
class CreditCardFraudModel(nn.Module):
    def __init__(self):
        super(CreditCardFraudModel, self).__init__()
        self.gated_layer1 = GatedLayer(29, 16)  # First gated layer with input size 29 and output size 16
        self.gated_layer2 = GatedLayer(16, 8)   # Second gated layer with input size 16 and output size 8
        self.output_layer = nn.Linear(8, 1)     # Output layer with input size 8 and output size 1

    def forward(self, x):
        x = self.gated_layer1(x)                # Pass through first gated layer
        x = self.gated_layer2(x)                # Pass through second gated layer
        x = self.output_layer(x)                # Pass through output layer
        return torch.sigmoid(x)                 # Apply sigmoid activation to get output in range [0, 1]